Run this code in a code cell
 before answering any of the question. This will create your knowledge
base and the FAISS index for this milestone.

In [3]:
!pip install faiss-cpu #install FAISS

In [4]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv('/content/train.csv')

In [5]:
print("Creating Knwoledge base...")
kb = []
for index, row in train.iterrows():
  correct_letter = row["answer"]
  kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index...")
model = SentenceTransformer('all-MiniLM-L6-v2')

kb_embeddings = model.encode(kb, show_progress_bar = False)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base successfully created....")

Creating Knwoledge base...
Loading embedding model and creating index...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created....


Zero-shot classifier for Q1, Q2, Q6:

In [6]:
from transformers import pipeline
zs = pipeline("zero-shot-classification", model = "facebook/bart-large-mnli")
row_150 = train.iloc[150]
prompt_150 = str(row_150["prompt"])
labels_150 = [str(row_150["A"]), str(row_150["B"]), str(row_150["C"]), str(row_150["D"]), str(row_150["E"])]
ans_150    = str(row_150[row_150["answer"]])

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Q1. Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points)

In [7]:
prediction = zs(prompt_150, candidate_labels = labels_150)

In [8]:
for idx, label in enumerate(prediction["labels"]):
  if label == ans_150:
    print(f"Probability score assigned to ground-truth correct option: {prediction["scores"][idx]:.3f}")

Probability score assigned to ground-truth correct option: 0.384


Q2. Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)?

In [9]:
prompt_150_embedding = model.encode(prompt_150)
prompt_150_embedding = prompt_150_embedding.reshape(1,-1)
distances, indices = index.search(prompt_150_embedding, k=10)
print(distances)

In [13]:
for i in range(len(indices[0])):
  if indices[0][i] == 150:
    print("Rank of the true correct document:", i+1)

Rank of the true correct document: 10


Q3. Take the top 10 documents retrieved by FAISS in the previous question. Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document?

In [18]:
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in indices[0]]

pairs = [[prompt_150,doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [20]:
print(ce_scores)
print(indices[0])

[4.6602316 4.6602316 4.737523  4.737523  4.687043  4.752577  4.752577
 4.752577  4.7072477 4.758512 ]
[ 663 1701 1269 1532  576  847 1693 1906  168  150]


In [32]:
scores = []
for i in range(len(indices[0])):
  scores.append((ce_scores[i], indices[0][i]))
scores.sort(reverse=True)
for i in range(len(scores)):
  if scores[i][1] == 150:
    print("Rank of the true correct document:" , i+1)


Rank of the true correct document: 1


Q4. Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: "Context: [concatenated_docs] Question: [prompt]". Tokenize this string using the bert-base-uncased tokenizer (without truncation). Exactly how many total tokens does this generate?

In [33]:
row_42 = train.iloc[42]
prompt_42 = str(row_42["prompt"])
labels_42 = [str(row_42["A"]), str(row_42["B"]), str(row_42["C"]), str(row_42["D"]), str(row_42["E"])]
answer_42 = str(row_42[row_42["answer"]])

In [39]:
prompt_42_embedding = model.encode(prompt_42).reshape(1,-1)
distances_5, indices_5 = index.search(prompt_42_embedding, k=5)
concatenated_docs = []
for doc in indices_5[0]:
  concatenated_docs.append(kb[doc])

concatenated_docs = " ".join(concatenated_docs)

system_prompt = f"Context: {concatenated_docs} Question: {prompt_42}"

In [45]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
input_tokens = tokenizer(system_prompt, truncation=False, return_tensors = "pt")

In [52]:
print(f"Number of tokens generated: {input_tokens["input_ids"].shape[1]}")

Number of tokens generated: 216


Q5. Retrieve the exact true document for row index 150 from your KB. Create a RAG string: "Context: [true_document] Question: [prompt]". Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places).